In [21]:
import os
from typing import Annotated, TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langgraph.graph import StateGraph, START, END

In [22]:
load_dotenv()

True

In [23]:
model = ChatGoogleGenerativeAI(model="gemini-3.8-flash")
wikipedia_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=2000))

In [24]:
class BlogState(TypedDict):
    topic: str              # Original user topic
    slug: str               # Generated Wikipedia search term
    wiki_content: str       # Retrieved Wikipedia text
    draft: str              # Generated blog draft
    critique: str           # Feedback or needed improvements
    revision_count: int

In [25]:
def get_content_string(response) -> str:
    content = response.content
    if isinstance(content, list):
        # Extract text blocks from the list format
        return "".join([item.get("text", "") if isinstance(item, dict) else str(item) for item in content]).strip()
    return str(content).strip()

In [26]:
def generate_slug_node(state: BlogState) -> BlogState:
    """Step 1: Convert the user's raw topic into a clean Wikipedia search query/slug."""
    topic = state["topic"]
    prompt = f"Convert the following blog topic into a concise, highly specific Wikipedia article title or search keyword. Return ONLY the search term: '{topic}'"
    response = model.invoke(prompt)
    slug = get_content_string(response)
    print(f"\n[Slug Generated]: {slug}")
    return {"slug": slug, "revision_count": 0}

In [27]:
def search_wikipedia_node(state: BlogState) -> BlogState:
    """Step 2: Search Wikipedia using the generated slug."""
    slug = state["slug"]
    try:
        wiki_result = wikipedia_tool.run(slug)
    except Exception as e:
        wiki_result = f"Error fetching Wikipedia article: {str(e)}"
    
    print(f"\n[Wikipedia Search Completed for]: {slug}")
    return {"wiki_content": wiki_result}

In [28]:
def draft_blog_node(state: BlogState) -> BlogState:
    """Step 3: Draft or revise the blog post using Wikipedia context."""
    topic = state["topic"]
    wiki_content = state["wiki_content"]
    draft = state.get("draft", "")
    critique = state.get("critique", "")
    
    if not draft:
        prompt = f"""Write an engaging, well-structured blog post about '{topic}' 
        based strictly on the following reference information:
        
        {wiki_content}
        """
    else:
        prompt = f"""Revise and improve the following blog draft about '{topic}' 
        based on this critique: '{critique}'.
        
        Original Draft:
        {draft}
        
        Reference Content:
        {wiki_content}
        """
        
    response = model.invoke(prompt)
    print("\n[Blog Draft Generated/Revised]")
    return {"draft": get_content_string(response)}

In [29]:
def evaluate_blog_node(state: BlogState) -> BlogState:
    """Step 4: Check if the blog needs improvement."""
    draft = state["draft"]
    revision_count = state.get("revision_count", 0)
    
    # Safeguard against endless loops
    if revision_count >= 2:
        return {"critique": "APPROVED"}

    prompt = f"""Review the following blog draft for clarity, depth, formatting, and accuracy.
    If it is good quality, respond with exactly: "APPROVED".
    If it needs improvement, briefly explain what needs to be fixed:
    
    {draft}
    """
    response = model.invoke(prompt)
    critique = get_content_string(response)
    print(f"\n[Evaluation Result]: {critique[:60]}...")
    
    return {"critique": critique, "revision_count": revision_count + 1}

In [30]:
def check_improvement_needed(state: BlogState) -> str:
    """Routing function based on evaluation feedback."""
    critique = state.get("critique", "")
    if "APPROVED" in critique or state.get("revision_count", 0) >= 2:
        return "end"
    return "improve"

In [31]:
workflow = StateGraph(BlogState)

# Add Nodes
workflow.add_node("generate_slug", generate_slug_node)
workflow.add_node("search_wikipedia", search_wikipedia_node)
workflow.add_node("draft_blog", draft_blog_node)
workflow.add_node("evaluate_blog", evaluate_blog_node)

# Define Edges
workflow.add_edge(START, "generate_slug")
workflow.add_edge("generate_slug", "search_wikipedia")
workflow.add_edge("search_wikipedia", "draft_blog")
workflow.add_edge("draft_blog", "evaluate_blog")

In [32]:
workflow.add_conditional_edges(
    "evaluate_blog",
    check_improvement_needed,
    {
        "improve": "draft_blog",
        "end": END
    }
)

# Compile Graph
app = workflow.compile()

In [33]:
initial_input = {"topic": "The history and impact of artificial intelligence"}
final_state = app.invoke(initial_input)

print("\n" + "="*40 + "\nFINAL PUBLISHED BLOG:\n" + "="*40)
print(final_state["draft"])


[Slug Generated]: History of artificial intelligence

[Wikipedia Search Completed for]: History of artificial intelligence

[Blog Draft Generated/Revised]

[Evaluation Result]: This text needs improvement because it is not a blog post; i...

[Blog Draft Generated/Revised]

[Evaluation Result]: APPROVED...

FINAL PUBLISHED BLOG:
# From Logic Engines to Neural Networks: The History and Impact of Artificial Intelligence

Once the exclusive domain of science fiction, artificial intelligence (AI) has become a defining force of the modern era. From the recommendation algorithms curating our music to neural networks diagnosing medical scans, AI influences how we work, communicate, and solve problems. 

Yet, today's breakthroughs did not emerge overnight. The story of artificial intelligence spans decades of bold ambition, disappointing setbacks, and sudden leaps forward. Here is a look at where AI came from—and how it is transforming our world.

---

## 1. The Dawn of Machine Minds (1950s–19